In [19]:
import logging
import os
import sys
from google.colab import drive

!pip install tensorboardX
from tensorboardX import SummaryWriter

# Настройка логирования
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s', stream=sys.stdout)
logger = logging.getLogger(__name__)

# Подготовка к импорту локальных python модулей
REPO_PATH = "/content/drive/MyDrive/bfu-lama"
if REPO_PATH not in sys.path: sys.path.append(REPO_PATH)
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
print(REPO_PATH)
!cd "{REPO_PATH}" && ls
# done: !cd "{REPO_PATH}" && git init
# !cd "{REPO_PATH}" && rm -rf .git


/content/drive/MyDrive/bfu-lama
 device_utils.py  'push to github.ipynb'   tenzorboard_monitor.ipynb
 main.py	   __pycache__		   utils_colab.py
 memory_utils.py   run_colab.ipynb	   utils_files.py


In [22]:
import utils_files, utils_colab
from env_config import GH_USER, PROJECT_NAME, GH_USER

utils_colab.git_pull(REPO_PATH)


### Отправка изменений на GitHub (Git Push)
В этом блоке выполняется конфигурация Git и отправка локальных изменений в удаленный репозиторий.

# Введите ваши данные или используйте userdata.get('GITHUB_TOKEN')
REPO_PATH = "/content/drive/MyDrive/3D_STA_SWGAIN"

REPO_FULL_NAME = "And0k/3D_STA_SWGAIN"


# Список всех локальных .py файлов в испльзуемом репозитории
python_files = utils_files.get_files_list(REPO_PATH)
print('\n'.join(python_files))
# Содержимое всех локальных .py файлов в используемом репозитории

utils_files.generate_markdown(REPO_PATH, python_files, output_file="repo_files.md")

TimeoutException: Requesting secret github_token timed out. Secrets can only be fetched when running from the Colab UI.


============================== Content of /device_utils.py START ==============================
# device_utils.py
import torch


def init_device():
    """
    Detect device once.
    Returns (device, IS_XLA, xm, parallel_loader_factory, optimizer_step, save_fn, is_master).
    """
    try:
        import torch_xla.core.xla_model as xm
        import torch_xla.distributed.parallel_loader as pl

        IS_XLA = True
        device = xm.xla_device()

        def parallel_loader_factory(loader, device):
            return pl.ParallelLoader(loader, [device]).per_device_loader(device)

        def optimizer_step(opt):
            xm.optimizer_step(opt, barrier=True)
            xm.mark_step()

        def save_fn(state, path):
            if xm.is_master_ordinal():
                xm.save(state, path)

        def is_master():
            return xm.is_master_ordinal()

    except Exception:
        # fallback CPU/GPU
        IS_XLA = False
        xm = None
        device = torch.device("

### Перезапуск обучения

In [ ]:
import memory_utils
import main
import importlib
import gc
import torch
import os
import logging
import sys

try:
    # Принудительная перезагрузка модулей для применения изменений путей
    importlib.reload(main)

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # Указываем абсолютный путь к чекпоинтам при запуске
    ckpt_path = os.path.join(repo_path, 'checkpoints')
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    # Включаем расширенную отладку чекпоинтов для поиска причины CheckpointError
    logging.info("Starting training with checkpoint debug mode enabled...")
    with torch.utils.checkpoint.set_checkpoint_debug_enabled(True):
        main.run_training(device=device, checkpoint_dir=ckpt_path, repo_path=repo_path)


    # Вызов функции синхронизации
    utils_colab.git_push_changes(
        REPO_PATH, f"{GH_USER}/{PROJECT_NAME}", github_user=GH_USER, msg="training finished"
    )
except Exception as e:
    logging.exception(f"CRITICAL: Training failed: {e}")
finally:
    sys.stdout.flush()

2026-05-23 15:36:01,486 - root - INFO - Starting training with checkpoint debug mode enabled...
2026-05-23 15:36:01,490 - root - INFO - Directory initialized: logs -> /content/drive/MyDrive/3D_STA_SWGAIN/logs
2026-05-23 15:36:01,491 - root - INFO - Directory initialized: checkpoints -> /content/drive/MyDrive/3D_STA_SWGAIN/checkpoints
2026-05-23 15:36:01,493 - root - INFO - Directory initialized: output -> /content/drive/MyDrive/3D_STA_SWGAIN/output
2026-05-23 15:36:01,508 - root - INFO - TensorBoard writer initialized at: /content/drive/MyDrive/3D_STA_SWGAIN/logs/260523
2026-05-23 15:36:01,817 - root - INFO - Critic initialized: base_dim=16, head_in=32
2026-05-23 15:36:01,824 - root - INFO - --- Starting Training | Mode: Zero-GP | G-Checkpointing: True ---


Epoch 1:   0%|          | 0/200 [00:00<?, ?it/s]

2026-05-23 15:36:08,415 - root - ERROR - CRITICAL: Training failed: Given groups=1, weight of size [16, 8, 3, 3, 3], expected input[1, 6, 32, 32, 32] to have 8 channels, but got 6 channels instead
Traceback (most recent call last):
  File "/tmp/ipykernel_130862/1496742.py", line 25, in <cell line: 0>
    main.run_training(device=device, checkpoint_dir=ckpt_path, repo_path=repo_path)
  File "/content/drive/MyDrive/3D_STA_SWGAIN/main.py", line 84, in run_training
    global_step = train_epoch(
                  ^^^^^^^^^^^^
  File "/content/drive/MyDrive/3D_STA_SWGAIN/training/trainer.py", line 31, in train_epoch
    # Apply first-order penalty
         ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1776, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1787, in _call_impl
    return forward_call(

In [12]:
import torch
import sys
from data.dataset import SyntheticOceanDataset

# 1. Проверка веса датасета
dataset = SyntheticOceanDataset(n_samples=200, T_window=4, Z=8, H=32, W=32, channels=2)
print(f"Dataset samples: {len(dataset)}")

# 2. Проверка размера одного батча
batch = dataset[0]
field = batch['field']
mask = batch['mask']

def get_size_mb(tensor):
    return (tensor.element_size() * tensor.nelement()) / 1024**2

print(f"Single field shape: {field.shape}, Size: {get_size_mb(field):.2f} MB")
print(f"Single mask shape: {mask.shape}, Size: {get_size_mb(mask):.2f} MB")

# 3. Эмуляция загрузки всех данных в RAM
total_data_ram = get_size_mb(field) * 200
print(f"Estimated total data RAM (if pre-loaded): {total_data_ram:.2f} MB")

Dataset samples: 200
Single field shape: torch.Size([4, 2, 8, 32, 32]), Size: 0.25 MB
Single mask shape: torch.Size([4, 1, 8, 32, 32]), Size: 0.12 MB
Estimated total data RAM (if pre-loaded): 50.00 MB
